# Moving Average vs Naive Baseline Audit

This notebook evaluates rolling moving average forecasting across window sizes $K \in \{1, 2, 3, 4, 5, 10\}$ and naive last-value persistence on synthetic stationary time series trials. The study investigates the impact of smoothing window parameters on forecasting accuracy, measuring Mean Squared Error (MSE) across diverse noise conditions.

In [ ]:
# Install dependencies (Colab-compatible conditional install)
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0', 'scikit-learn==1.6.1')

In [ ]:
# Imports
import json
import os
import urllib.request
import numpy as np
import matplotlib.pyplot as plt

# NumPy 2.0 compatibility shims if needed
if not hasattr(np, "alltrue"): np.alltrue = np.all
if not hasattr(np, "sometrue"): np.sometrue = np.any
if not hasattr(np, "product"): np.product = np.prod

In [ ]:
# Data loading helper with GitHub URL and local fallback
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-29c492-empirical-audit-of-moving-average-baseli/main/round-2/experiment-1/demo/mini_demo_data.json"

def load_data():
    try:
        print(f"Attempting to load data from GitHub: {GITHUB_DATA_URL}")
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        print(f"GitHub load failed ({e}), falling back to local file.")
    
    for local_path in ["mini_demo_data.json", "mini_data_out.json"]:
        if os.path.exists(local_path):
            print(f"Loading from local file: {local_path}")
            with open(local_path, 'r') as f:
                return json.load(f)
                
    raise FileNotFoundError("Could not load mini_demo_data.json from GitHub or local path.")

data = load_data()

In [ ]:
# Configuration cell
K_VALUES = [1, 2, 3, 4, 5, 10]
OUTPUT_PATH = "method_out.json"

### Run Forecasting Evaluation

We iterate over all loaded time series trials, computing the Mean Squared Error (MSE) for naive last-value persistence as well as moving average forecasts across various window sizes $K$.

In [ ]:
examples = data['datasets'][0]['examples']
print(f"Total trials loaded: {len(examples)}")

new_examples = []

for i, ex in enumerate(examples):
    series = np.array(json.loads(ex['input']))
    true_mean = float(ex['output'])
    length = ex['metadata_length']
    noise_var = ex['metadata_noise_variance']
    trial_id = ex['metadata_trial_id']
    
    if len(series) > 1:
        actuals = series[1:]
        naive_preds = series[:-1]
        naive_mse = float(np.mean((actuals - naive_preds) ** 2))
    else:
        naive_mse = 0.0
        
    ex_out = {
        "input": ex['input'],
        "output": ex['output'],
        "metadata_trial_id": trial_id,
        "metadata_length": length,
        "metadata_noise_variance": noise_var,
        "predict_naive": str(naive_mse)
    }
    
    for k in K_VALUES:
        if len(series) >= k + 1:
            actuals = series[k:]
            preds = []
            for t in range(k, len(series)):
                window = series[t-k:t]
                preds.append(np.mean(window))
            preds = np.array(preds)
            ma_mse = float(np.mean((actuals - preds) ** 2))
        else:
            ma_mse = naive_mse
        ex_out[f"predict_MA_K_{k}"] = str(ma_mse)
        
    new_examples.append(ex_out)

final_output = {
    "datasets": [
        {
            "dataset": data['datasets'][0]['dataset'],
            "examples": new_examples
        }
    ]
}

print("Saving results to", OUTPUT_PATH)
with open(OUTPUT_PATH, 'w') as f:
    json.dump(final_output, f, indent=2)
print("Evaluation completed successfully.")

### Visualization and Summary

We summarize the performance across models by averaging MSE over all trials for each forecasting method, and plot the comparative MSE.

In [ ]:
# Aggregate MSE across trials
methods = ['predict_naive'] + [f'predict_MA_K_{k}' for k in K_VALUES]
mean_mses = {}

for m in methods:
    vals = [float(ex[m]) for ex in new_examples]
    mean_mses[m] = np.mean(vals)

print("Average MSE by Method:")
for m, val in mean_mses.items():
    print(f"  {m}: {val:.4f}")

# Plotting
plt.figure(figsize=(8, 4))
labels = ['Naive'] + [f'MA (K={k})' for k in K_VALUES]
values = list(mean_mses.values())

plt.bar(labels, values, color=['gray'] + ['skyblue']*len(K_VALUES))
plt.ylabel('Mean Squared Error (MSE)')
plt.title('Forecasting Performance: Naive vs Moving Average (by Window Size K)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()